Linear Regression & MLE: Proves that minimizing sum-of-squared errors is equivalent to Maximum Likelihood Estimation under Gaussian observation noise.   
Ridge Regression ($L_2$) & MAP: Shows that adding a squared norm penalty ($\lambda \Vert{}w\Vert{}_2^2$) corresponds to MAP estimation under a zero-mean Gaussian parameter prior.   
Lasso Regression ($L_1$): Demonstrates how absolute value penalties ($\lambda \Vert{}w\Vert{}_1$) enforce sparse weight vectors, performing feature selection in high-dimensional settings (e.g., genomics).   

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# Set seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully.")

Step 1: Synthesizing High-Dimensional Biological Data
In scientific and genomic contexts, we often encounter settings where the number of features (e.g., genetic markers or gene expression levels) approaches or exceeds the number of observations ($p \approx n$ or $p > n$).   We generate a synthetic dataset with $n=60$ samples and $p=40$ features. Only 5 features have a true non-zero effect on the phenotype $Y$; the remaining 35 features represent uninformative biological noise

In [ ]:
n_samples = 60
n_features = 40

# Generate random feature matrix X
X = np.random.normal(0, 1, size=(n_samples, n_features))

# True parameter vector w*: sparse with only 5 active features
true_w = np.zeros(n_features)
true_w[:5] = [3.5, -2.0, 4.0, -1.5, 2.5]

# Generate targets Y = Xw* + noise
noise = np.random.normal(0, 1.5, size=n_samples)
Y = X @ true_w + noise

# Train/Test Split
split = 40
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

print(f"Dataset created: {n_samples} samples, {n_features} features.")
print(f"True non-zero weights: {true_w[:5]}")

Step 2: Implementing Ordinary Least Squares, Ridge, and Lasso
We fit three fundamental regression models to the training data:   Ordinary Least Squares (OLS): Computes unregularized closed-form estimates $\hat{w} = (X^\top X)^{-1}X^\top y$. Highly vulnerable to overfitting noise when $p$ is large relative to $n$.   Ridge Regression ($L_2$ Regularization): Solves $\hat{w} = (X^\top X + \lambda I)^{-1}X^\top y$, shrinking coefficients toward zero to stabilize variance.   Lasso Regression ($L_1$ Regularization): Solves $\min_w \Vert{}y - Xw\Vert{}_2^2 + \lambda \Vert{}w\Vert{}_1$, driving irrelevant coefficients to exactly zero

In [ ]:
# 1. Ordinary Least Squares (OLS)
ols = LinearRegression()
ols.fit(X_train, Y_train)

# 2. Ridge Regression (L2 penalty)
ridge = Ridge(alpha=10.0)
ridge.fit(X_train, Y_train)

# 3. Lasso Regression (L1 penalty)
lasso = Lasso(alpha=0.3)
lasso.fit(X_train, Y_train)

# Plotting weight estimates against ground truth
plt.figure(figsize=(12, 5))
plt.plot(true_w, 'k--', label='True Weights (w*)', linewidth=2)
plt.plot(ols.coef_, 'o-', label='OLS (Unregularized MLE)', alpha=0.7)
plt.plot(ridge.coef_, 's-', label='Ridge (L2 MAP)', alpha=0.7)
plt.plot(lasso.coef_, '^--', label='Lasso (L1 Feature Selection)', alpha=0.9)

plt.title('Weight Recovery Comparison Across Regression Models', fontsize=14)
plt.xlabel('Feature Index', fontsize=12)
plt.ylabel('Estimated Coefficient Value', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()

# Print MSE on Test Set
print(f"OLS Test MSE:   {mean_squared_error(Y_test, ols.predict(X_test)):.4f}")
print(f"Ridge Test MSE: {mean_squared_error(Y_test, ridge.predict(X_test)):.4f}")
print(f"Lasso Test MSE: {mean_squared_error(Y_test, lasso.predict(X_test)):.4f}")

Step 3: Regularization Paths & Feature Selection
The strength of regularization is controlled by the hyperparameter $\lambda$ ($\alpha$ in scikit-learn):   Ridge Path: As $\lambda$ increases, all feature weights continuously shrink toward zero, but none reach exactly zero.   Lasso Path: As $\lambda$ increases, weights hit zero at distinct thresholds, effectively eliminating noisy features and performing automated model selection.

In [ ]:
alphas = np.logspace(-2, 2, 100)

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    r = Ridge(alpha=a).fit(X_train, Y_train)
    l = Lasso(alpha=a, max_iter=2000).fit(X_train, Y_train)
    ridge_coefs.append(r.coef_)
    lasso_coefs.append(l.coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge Path
axes[0].plot(alphas, ridge_coefs)
axes[0].set_xscale('log')
axes[0].set_title('Ridge Path (L2): Continuous Shrinkage', fontsize=13)
axes[0].set_xlabel('Regularization Strength (λ / alpha)', fontsize=11)
axes[0].set_ylabel('Coefficient Values', fontsize=11)

# Lasso Path
axes[1].plot(alphas, lasso_coefs)
axes[1].set_xscale('log')
axes[1].set_title('Lasso Path (L1): Feature Sparsity & Selection', fontsize=13)
axes[1].set_xlabel('Regularization Strength (λ / alpha)', fontsize=11)
axes[1].set_ylabel('Coefficient Values', fontsize=11)

plt.tight_layout()
plt.show()